In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
gdf = gpd.read_file('311_Encampment_Reports%2C_City_of_San_Diego%2C_2021.geojson')

In [ ]:
gdf.columns


In [ ]:
docs = gdf.public_des.tolist()
docs[0:5]

## Data cleaning
- removing stop words
- only keeping records that have the regular expression that includes "tent" and "camp"

In [ ]:
import re

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# regular expression pattern for encampment-related keywords
encampment_pattern = re.compile(r'\b\w*(tent|camp)\w*\b', re.IGNORECASE)

# filter docs to include only those related to encampments
filtered_docs = [doc for doc in docs if encampment_pattern.search(doc)]
print(f"Original number of documents: {len(docs)}")
print(f"Number of encampment-related documents: {len(filtered_docs)}")

# remove stopwords from the filtered documents
cleaned_docs = []
for doc in filtered_docs:
    # Remove stopwords by keeping only words that are not in the stopword set
    cleaned_doc = ' '.join([word for word in doc.split() if word.lower() not in ENGLISH_STOP_WORDS])
    cleaned_docs.append(cleaned_doc)

print(f"Number of documents after stopword removal: {len(cleaned_docs)}")


In [ ]:
cleaned_docs[0:5]

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Combine all documents into a single string for the word cloud
text = ' '.join(cleaned_docs)

# Create and configure the WordCloud object
wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis').generate(text)

# Display the WordCloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')  # Turn off the axis
plt.show()

## Build an LDA Model

this is a baseline LDA model that is used for initial testing

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from gensim import corpora
# from gensim.models.ldamodel import LdaModel

# # convert the cleaned docs into TF-IDF format
# tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2)  
# tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_docs)

# # convert TF-IDF matrix to a corpus for gensim LDA model
# # get the words from TF-IDF vectorizer
# terms = tfidf_vectorizer.get_feature_names_out()

# # convert the TF-IDF sparse matrix to dense format
# dense_tfidf_matrix = tfidf_matrix.todense()

# # create a dictionary and a gensim corpus
# corpus = [[(i, dense_tfidf_matrix[row, i]) for i in range(len(terms)) if dense_tfidf_matrix[row, i] > 0]
#           for row in range(dense_tfidf_matrix.shape[0])]

# dictionary = corpora.Dictionary([terms])

# # build the LDA model using the TF-IDF weighted corpus
# def build_lda_model(corpus, dictionary, num_topics=5, alpha='auto', beta='auto', passes=10):
#     lda_model = LdaModel(corpus=corpus,
#                          id2word=dictionary,
#                          num_topics=num_topics,      
#                          random_state=100,
#                          update_every=1,
#                          chunksize=100,
#                          passes=passes,              
#                          alpha=alpha,                
#                          eta=beta,                   
#                          per_word_topics=True)
#     return lda_model

In [ ]:
# # function to extract topics and format them into a df
# def lda_to_dataframe(lda_model, num_words=10):
#     topics = lda_model.show_topics(formatted=False, num_words=num_words)

#     topic_dict = {'Topic': [], 'Top Words': []}

#     for topic_num, word_probs in topics:
#         words = ", ".join([f"{word} ({prob:.2f})" for word, prob in word_probs])  
#         topic_dict['Topic'].append(f"Topic {topic_num + 1}")
#         topic_dict['Top Words'].append(words)

#     df_topics = pd.DataFrame(topic_dict)
#     return df_topics

In [ ]:
# lda_model_tfidf = build_lda_model(corpus, dictionary, num_topics=4, alpha='auto', beta='auto', passes=10)

# df_topics_tfidf_5 = lda_to_dataframe(lda_model_tfidf, num_words=10)

# df_topics_tfidf_5

In [ ]:
# !pip install pyLDAvis

In [ ]:
# import pyLDAvis
# import pyLDAvis.gensim_models as gensimvis
# import gensim

# lda_vis = gensimvis.prepare(lda_model_tfidf, corpus, dictionary)

# pyLDAvis.display(lda_vis)

# # Optionally, save it as an HTML file
# pyLDAvis.save_html(lda_vis, 'lda_topics.html')


In [ ]:
# from google.colab import files

# # Download the HTML file to your local system
# files.download('lda_topics.html')

1. do coherence score testing for different LDA models
2. run again while taking out the camp and tent words

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from gensim import corpora
# from gensim.models.ldamodel import LdaModel
# from gensim.models import CoherenceModel

# # convert the cleaned documents into TF-IDF format
# tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2)
# tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_docs)

# # convert TF-IDF matrix to gensim-compatible corpus
# terms = tfidf_vectorizer.get_feature_names_out()
# dense_tfidf_matrix = tfidf_matrix.todense()
# corpus = [[(i, dense_tfidf_matrix[row, i]) for i in range(len(terms)) if dense_tfidf_matrix[row, i] > 0]
#           for row in range(dense_tfidf_matrix.shape[0])]

# dictionary = corpora.Dictionary([terms])

# # remove empty and short docs from cleaned_docs
# min_word_count = 3
# filtered_cleaned_docs = [doc for doc in cleaned_docs if len(doc.split()) >= min_word_count]

# # tokenize the filtered docs
# tokenized_docs = [doc.split() for doc in filtered_cleaned_docs]


# # **THIS IS TO TUNE TO FIND THE BEST HYPERPARAMETERS FOR THE MODEL**

# # define ranges for num_topics, alpha, and beta
# num_topics_range = [3, 4, 5]  
# alpha_values = [0.01, 0.1]  
# beta_values = ['symmetric', 0.1]   

# model_results = []
# best_coherence = 0  
# best_model = None   

# for num_topics in num_topics_range:
#     for alpha in alpha_values:
#         for beta in beta_values:
#             # build LDA model with the current set of parameters
#             lda_model = LdaModel(corpus=corpus,
#                                  id2word=dictionary,
#                                  num_topics=num_topics,
#                                  alpha=alpha,
#                                  eta=beta,
#                                  passes=50,
#                                  iterations=100,
#                                  random_state=100)

#             # calculate coherence score for the model 
#             coherence_model_lda = CoherenceModel(model=lda_model, texts=tokenized_docs, dictionary=dictionary, coherence='u_mass')
#             coherence_score = coherence_model_lda.get_coherence()

#             # store the model results in the list as a tuple 
#             model_results.append((coherence_score, num_topics, alpha, beta, lda_model))

#             # check if this model has the best coherence score so far
#             if coherence_score > best_coherence or best_coherence == 0:
#                 best_coherence = coherence_score
#                 best_model = lda_model  

# model_results.sort(reverse=True, key=lambda x: x[0])

# # Print the top 5 models
# print("\nTop 5 Models with Best Coherence Scores:")
# for i, result in enumerate(model_results[:5]):
#     print(f"Rank {i+1}: Coherence Score: {result[0]:.4f} | Num Topics: {result[1]} | Alpha: {result[2]} | Beta: {result[3]}")

# # save the best model to disk or use it in future analysis
# if best_model:
#     best_model.save('best_lda_model')
#     print(f'\nBest Model saved with Coherence Score: {best_coherence}')
# else:
#     print("No valid model found with a coherence score.")


In [ ]:
# for i, topic in best_model.show_topics(formatted=False, num_words=10):
#     print(f"Topic {i+1}: {[word for word, prob in topic]}")


In [ ]:
# import pyLDAvis.gensim_models as gensimvis
# import pyLDAvis

# # visualize the best model
# lda_vis = gensimvis.prepare(best_model, corpus, dictionary)
# pyLDAvis.display(lda_vis)


## Now removing camp words so they don't dominate the topics to see the differences

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim import corpora
from gensim.models.ldamodel import LdaModel
from gensim.models import CoherenceModel

camp_pattern = re.compile(r'\b\w*(tent|camp)\w*\b', re.IGNORECASE)

# remove "camp" related terms from each doc in cleaned_docs
cleaned_docs_no_camp = [camp_pattern.sub('', doc).strip() for doc in cleaned_docs]


In [ ]:
print(len(cleaned_docs_no_camp))

In [ ]:
# convert the cleaned docs into TF-IDF format
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_docs_no_camp)

# convert TF-IDF matrix to a gensim-compatible corpus
terms = tfidf_vectorizer.get_feature_names_out()
dense_tfidf_matrix = tfidf_matrix.todense()
corpus = [[(i, dense_tfidf_matrix[row, i]) for i in range(len(terms)) if dense_tfidf_matrix[row, i] > 0]
          for row in range(dense_tfidf_matrix.shape[0])]

# create a gensim dictionary from the terms
dictionary = corpora.Dictionary([terms])

num_topics_range = [3, 4, 5] 
alpha_values = [0.01, 0.1]  
beta_values = [0.1, 'symmetric']   

model_results = []
best_coherence = 0  
best_model = None   

# iterate over num_topics, alpha, and beta (eta) to find the best model
for num_topics in num_topics_range:
    for alpha in alpha_values:
        for beta in beta_values:
            # build LDA model with the current set of parameters
            lda_model = LdaModel(corpus=corpus,
                                 id2word=dictionary,
                                 num_topics=num_topics,
                                 alpha=alpha,
                                 eta=beta,
                                 passes=50,        
                                 iterations=100,   
                                 random_state=100)

            # calculate coherence score for the model 
            tokenized_docs_no_camp = [doc.split() for doc in cleaned_docs_no_camp if doc.strip()]
            coherence_model_lda = CoherenceModel(model=lda_model, texts=tokenized_docs_no_camp, dictionary=dictionary, coherence='u_mass')
            coherence_score = coherence_model_lda.get_coherence()

            model_results.append((coherence_score, num_topics, alpha, beta, lda_model))

            if coherence_score > best_coherence or best_coherence == 0:
                best_coherence = coherence_score
                best_model = lda_model  

model_results.sort(reverse=True, key=lambda x: x[0])

# print the top 5 models
print("\nTop 5 Models with Best Coherence Scores:")
for i, result in enumerate(model_results[:5]):
    print(f"Rank {i+1}: Coherence Score: {result[0]:.4f} | Num Topics: {result[1]} | Alpha: {result[2]} | Beta: {result[3]}")

# save the best model to disk or use it in future analysis
if best_model:
    best_model.save('best_lda_model_no_camp')
    print(f'\nBest Model saved with Coherence Score: {best_coherence}')
else:
    print("No valid model found with a coherence score.")


In [ ]:
for i, topic in best_model.show_topics(formatted=False, num_words=10):
    print(f"Topic {i+1}: {[word for word, prob in topic]}")


In [ ]:
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

# visualize the best model
lda_vis = gensimvis.prepare(best_model, corpus, dictionary)
pyLDAvis.display(lda_vis)


# now need to apply all this to the overall dataframe and create KDE

In [ ]:
filtered_gdf = gdf[gdf['public_des'].isin(filtered_docs)].copy()

# verify that filtered_gdf has the same length as the filtered docs 
print(f"Number of records in filtered_gdf: {len(filtered_gdf)}")  


In [ ]:
import numpy as np
from sklearn.neighbors import KernelDensity
import matplotlib.pyplot as plt
import contextily as ctx
import geopandas as gpd

# assign dominant topics to each document
topic_assignments = []
for bow in corpus:
    topics = best_model.get_document_topics(bow)
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    topic_assignments.append(dominant_topic)

# add topic assignments to the existing gdf
filtered_gdf['topic'] = topic_assignments


In [ ]:
filtered_gdf.head()

In [ ]:
# set global boundaries for consistent KDE grid across all topics
x_min_global, y_min_global, x_max_global, y_max_global = gdf.total_bounds

# function to create KDE maps with a larger title and color legend
def plot_kernel_density(gdf, title, bandwidth=0.005):
    gdf = gdf[gdf.geometry.notnull()]
    
    coords = np.array([[point.x, point.y] for point in gdf.geometry])
    
    kde = KernelDensity(bandwidth=bandwidth, kernel='gaussian')
    kde.fit(coords)
    
    x_grid = np.linspace(x_min_global, x_max_global, 500)
    y_grid = np.linspace(y_min_global, y_max_global, 500)
    x_mesh, y_mesh = np.meshgrid(x_grid, y_grid)
    xy_sample = np.vstack([x_mesh.ravel(), y_mesh.ravel()]).T

    z = np.exp(kde.score_samples(xy_sample)).reshape(x_mesh.shape)

    fig, ax = plt.subplots(figsize=(10, 10))
    kde_plot = ax.imshow(z, origin='lower', cmap='inferno', extent=[x_min_global, x_max_global, y_min_global, y_max_global], alpha=1)
    
    ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.4)
    
    plt.title(title, fontsize=16)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    
    cbar = plt.colorbar(kde_plot, ax=ax, fraction=0.036, pad=0.04)
    cbar.set_label('Density', rotation=270, labelpad=15, fontsize=12)
    
    plt.show()


In [ ]:
# create dataframe to display words in each topic as well
num_words = 5

for topic_num in sorted(filtered_gdf['topic'].unique()):
    topic_gdf = filtered_gdf[filtered_gdf['topic'] == topic_num]
    
    plot_kernel_density(topic_gdf, f"Topic {topic_num+1} / 2021")
    
    topic_words = best_model.show_topic(topic_num, topn=num_words)
    words_df = pd.DataFrame(topic_words, columns=["Word", "Probability"])
    
    words_df['Probability'] = words_df['Probability'].apply(lambda x: f"{x:.3f}")

    fig, ax = plt.subplots(figsize=(5, 2))  
    ax.axis('off')  
    table = ax.table(cellText=words_df.values, colLabels=words_df.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.auto_set_column_width(col=list(range(len(words_df.columns))))

    plt.savefig(f"Topic_{topic_num+1}_2021_words.png", bbox_inches='tight', dpi=300)
    plt.close(fig)

    print(f"Saved top words for Topic {topic_num+1} as PNG.")
